In [1]:
from __future__ import annotations

import csv
import re
import unicodedata
import warnings
from pathlib import Path
from typing import List

import geopandas as gpd
import numpy as np
import pandas as pd
from shapely import wkt
from shapely.ops import unary_union

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

In [2]:
ROOT = Path('/Users/ivanr/Developer/ai-vineyard-productivity')
DATA_DIR = ROOT / 'data' / 'raw'
OUT_DIR = ROOT / 'data' / 'datasets' / 'raw'
OUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    'innovi': {
        'book': DATA_DIR / 'Viñedos_Innovi_corrected (1).xlsx',
        'sigpac': DATA_DIR / 'Viñedos_Innovi_corrected_sigpac (1).xlsx',
    },
    'tactic': {
        'book': DATA_DIR / 'Tactic_plantilla_viñedo_CWP_corrected (1).xlsx',
        'sigpac': DATA_DIR / 'Tactic_plantilla_viñedo_CWP_corrected_sigpac (1).xlsx',
    },
}

FULL_AOI = Path("/Users/ivanr/Developer/ai-vineyard-productivity/data/raw/Full_aoi_table.xlsx")

# SIGPAC CRS for Catalonia: ETRS89 / UTM zone 31N
SRC_EPSG = 25831
DST_EPSG = 3857

# SIGPAC join keys
KEY_PARCELA = ['provincia', 'municipio', 'agregado', 'zona', 'poligono', 'parcela']
KEY_RECINTO = KEY_PARCELA + ['recinto']

for k, v in FILES.items():
    print(f'{k}: book={v["book"].exists()}, sigpac={v["sigpac"].exists()}')

innovi: book=True, sigpac=True
tactic: book=True, sigpac=True


## Helper functions


In [3]:
column_map_en = {
    "id_lugar": "place_id",
    "fecha_inicio_cosecha": "harvest_start_date",
    "fecha_fin_cosecha": "harvest_date",
    "produccion_kg": "production_kg",
    "grado_alcoholico": "alcohol_degree",
    "tipo_de_certificacion_manejo": "certification",
    "variedad_comercial_principal": "variety",
    "secano": "dry_farming"
}
cols_to_drop = [
    'densidad_plantacion_arboles_ha',
    'sistema_de_conduccion',
    'producto',
    'fecha_de_plantacion',
    'portainjerto',
    'marco_plantacion_mxm',
    'cultivo',
    'fecha_inicio_cosecha'
]
dtype_map = {
    'place_id': str,
    'harvest_date': 'datetime64[ns]',
    'production_kg': int,
    'alcohol_degree': float,
    'certification': 'category',
    'variety': 'category',
    'dry_farming': 'category'
}

In [4]:
def normalize_col(col: str) -> str:
    """Normalize column name: lowercase, ASCII, underscored."""
    col = str(col).strip().lower()
    col = unicodedata.normalize('NFKD', col).encode('ascii', 'ignore').decode('ascii')
    col = re.sub(r'[^a-z0-9]+', '_', col)
    return re.sub(r'_+', '_', col).strip('_')


def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Apply normalize_col to all column names."""
    out = df.copy()
    out.columns = [normalize_col(c) for c in out.columns]
    return out

## Load excels


In [5]:
def load_all_sheets(filepath: Path) -> dict[str, pd.DataFrame]:
    """Load every sheet from an Excel file with normalized column names."""
    xls = pd.ExcelFile(filepath)
    return {
        normalize_col(s): clean_columns(pd.read_excel(filepath, sheet_name=s))
        for s in xls.sheet_names
    }


sheets = {}
for source, paths in FILES.items():
    book = load_all_sheets(paths['book'])
    sig = load_all_sheets(paths['sigpac'])
    sheets[source] = {'book': book, 'sigpac': sig}

    print(f'\n--- {source.upper()} ---')
    for section in ('book', 'sigpac'):
        for name, df in sheets[source][section].items():
            print(f'  [{section}] {name}: {df.shape}')


aoi_df = load_all_sheets(FULL_AOI)['sheet_1']


--- INNOVI ---
  [book] id_lugar: (8341, 13)
  [book] finca_vinedo: (5262, 11)
  [book] produccion: (21388, 15)
  [book] fenologia: (772, 1)
  [book] fitosanitarios: (10742, 8)
  [book] fertilizantes: (1978, 14)
  [book] series_temporales: (27, 7)
  [sigpac] parcelas_sigpac: (3498, 7)
  [sigpac] recintos_sigpac: (7651, 8)

--- TACTIC ---
  [book] id_lugar: (15, 13)
  [book] finca_vinedo: (7, 11)
  [book] produccion: (35, 15)
  [book] fitosanitarios: (87, 8)
  [book] fertilizantes: (28, 14)
  [sigpac] parcelas_sigpac: (3, 7)
  [sigpac] recintos_sigpac: (15, 8)


In [6]:
raw_tactic = sheets['tactic']['book']['produccion']
raw_innovi = sheets['innovi']['book']['produccion']

## Merge and clean data

In [25]:
def merge_sheets(base: pd.DataFrame, book: dict[str, pd.DataFrame]) -> pd.DataFrame:
    for name, df in book.items():
        if name in ["finca_vinedo"]:
            cols = [c for c in df.columns if c != 'id_lugar']
            base = base.merge(df[['id_lugar'] + cols], on='id_lugar', how='left')

    return base

def clean_produccion(df: pd.DataFrame) -> pd.DataFrame:
    df = df.dropna(axis=1, how='all')
    if 'destino_del_cultivo' in df.columns:
        df = df.drop(columns=['destino_del_cultivo'])       
 
    return df

def normalize_place_id(place_id: str) -> List[str]:
    ids = [x.strip() for x in place_id.split(",")]
    ids = sorted(ids)
    return ",".join(ids)

def aggregate_duplicates_same_period(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df
        .assign(place_id_norm=df["place_id"].apply(normalize_place_id))
        .assign(year=df['harvest_date'].apply(lambda x: x.year))
        .groupby(
            ["place_id_norm", "year"],
            as_index=False
        )
        .agg({
            "production_kg": "sum",
            "alcohol_degree": "mean",
            "harvest_date": lambda x: df.loc[x.index, "harvest_date"].loc[
                df.loc[x.index, "production_kg"].idxmax()
            ]
        })
    ).rename(columns={"place_id_norm": "place_id"})

# Agrupar localizaciones
def aggregate_places_df(ref_df: pd.DataFrame, places_df: pd.DataFrame) -> pd.DataFrame:
    geo_cols = [
        "provincia",
        "municipio",
        "poligono",
        "parcela",
        "recinto"
    ]
    rows = []

    for place_id_str in ref_df["place_id"].unique():

        ids = [x.strip() for x in place_id_str.split(",")]

        subset = places_df[places_df["id_lugar"].isin(ids)]

        superficie = subset["superficie_ha"].sum()

        geo_data = {
            col: subset[col].astype(str).tolist()
            for col in geo_cols
        }

        rows.append({
            "place_id": place_id_str,
            "superficie_ha": superficie,
            **geo_data
        })
        
    return ref_df.merge(
        pd.DataFrame(rows),
        on="place_id",
        how="left"
    )

def assign_geometry(ref_df: pd.DataFrame, aoi_df: pd.DataFrame) -> pd.DataFrame:
    """
    For each row in ref_df, look up matching rows in aoi_df using
    (provincia, municipio, poligono, parcela, recinto) and merge their geometries.
    """
    # Build a lookup index on aoi_df keyed by (prov, mun, pol, par, rec)
    aoi_lookup = {}
    for idx, row in aoi_df.iterrows():
        key = (row["provincia"], row["municipio"], row["poligono"], row["parcela"], row["recinto"])
        # print(key)
        aoi_lookup.setdefault(key, []).append(row["geometry"])
    # print("----")
    geometries = []
    for _, row in ref_df.iterrows():
        geoms = []
        for prov, mun, pol, par, rec in zip(
            row["provincia"], row["municipio"], row["poligono"], row["parcela"], row["recinto"]
        ):
            key = (int(prov), int(mun), int(pol), int(par), int(rec))
            # print(key)
            if key in aoi_lookup:
                geoms.extend(aoi_lookup[key])

        if geoms:
            parsed = [wkt.loads(g) if isinstance(g, str) else g for g in geoms]
            geometries.append(unary_union(parsed))
        else:
            geometries.append(None)

    ref_df = ref_df.copy()
    ref_df["geometry"] = geometries
    return ref_df

def clean_location_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Count recintos
    df["total_recintos"] = df["recinto"].apply(len)

    # Deduplicate lists where all values are the same
    for col in ["provincia", "municipio", "poligono", "parcela", "recinto"]:
        df[col] = df[col].apply(lambda x: [x[0]] if len(set(x)) == 1 else x)

    return df

def compute_spatial_features(df: pd.DataFrame, crs: str = "EPSG:3857") -> gpd.GeoDataFrame:
    gdf = gpd.GeoDataFrame(df, geometry="geometry", crs=crs)
    gdf["area_ha"] = gdf.geometry.area / 10_000
    gdf["yield_kg_ha"] = gdf["production_kg"] / gdf["area_ha"]
    return gdf.round(2)

def rename_and_order_columns(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    gdf = gdf.rename(columns={
        "place_id": "parcel_id",
        "superficie_ha": "declared_area_ha",
        "provincia": "province",
        "municipio": "municipality",
        "poligono": "polygon",
        "parcela": "parcel",
        "recinto": "enclosure",
        "total_recintos": "total_enclosures",
    })

    column_order = [
        "parcel_id",
        "year",
        "harvest_date",
        "province",
        "municipality",
        "polygon",
        "parcel",
        "enclosure",
        "total_enclosures",
        "declared_area_ha",
        "area_ha",
        "production_kg",
        "yield_kg_ha",
        "alcohol_degree",
        "geometry",
    ]

    return gdf[column_order]


In [26]:
# Innovi
produccion_innovi = clean_produccion(sheets['innovi']['book']['produccion'])
innovi_df = merge_sheets(produccion_innovi, sheets['innovi']['book'])
innovi_df = innovi_df.drop(columns=cols_to_drop)
innovi_df = innovi_df.rename(columns=column_map_en)
innovi_df = innovi_df.astype(dtype_map, errors='ignore')
innovi_df = aggregate_duplicates_same_period(innovi_df) # Producciones de la misma finca el mismo año agrupar producciones
innovi_df = aggregate_places_df(innovi_df, sheets['innovi']['book']['id_lugar'])
innovi_df = assign_geometry(innovi_df, aoi_df)
innovi_df = clean_location_columns(innovi_df)
innovi_df = compute_spatial_features(innovi_df)
innovi_df = rename_and_order_columns(innovi_df)
final_innovi_df = innovi_df[innovi_df.geometry.notnull()]


print(final_innovi_df.shape)
print(final_innovi_df.head(1))

# Tactic
produccion_tactic = clean_produccion(sheets['tactic']['book']['produccion'])
tactic_df = merge_sheets(produccion_tactic, sheets['innovi']['book'])
tactic_df = tactic_df.drop(columns=cols_to_drop)
tactic_df = tactic_df.rename(columns=column_map_en)
tactic_df = tactic_df.astype(dtype_map, errors='ignore')
tactic_df = aggregate_duplicates_same_period(tactic_df) # Producciones de la misma finca el mismo año agrupar producciones
tactic_df = aggregate_places_df(tactic_df, sheets['tactic']['book']['id_lugar']) # 
tactic_df = assign_geometry(tactic_df, aoi_df)
tactic_df = clean_location_columns(tactic_df)
tactic_df = compute_spatial_features(tactic_df)
tactic_df = rename_and_order_columns(tactic_df)
final_tactic_df = tactic_df[tactic_df.geometry.notnull()]


print(tactic_df.shape)
print(tactic_df.head(1))

(11970, 15)
    parcel_id  year harvest_date province municipality polygon parcel enclosure  total_enclosures  declared_area_ha  area_ha  production_kg  yield_kg_ha  alcohol_degree  \
0  L1100,L194  2021   2021-09-22      [8]         [13]     [4]   [49]   [6, 10]                 2              1.19     2.12           6734      3180.32            11.4   

                                            geometry  
0  MULTIPOLYGON (((197985.277 5068113.845, 197898...  
(35, 15)
  parcel_id  year harvest_date province municipality polygon parcel enclosure  total_enclosures  declared_area_ha  area_ha  production_kg  yield_kg_ha  alcohol_degree  \
0        L1  2021   2021-08-30     [43]         [22]     [2]   [82]       [1]                 1              0.39     0.69           2180      3167.85            14.0   

                                            geometry  
0  POLYGON ((37985.287 5023205.192, 37985.735 502...  


In [27]:
def export_to_csv(gdf: gpd.GeoDataFrame, path: str) -> None:
    df = gdf.copy()
    # Ensure list columns have consistent int values
    list_cols = ["province", "municipality", "polygon", "parcel", "enclosure"]
    for col in list_cols:
        df[col] = df[col].apply(lambda x: [int(v) for v in x] if isinstance(x, list) else x)
        df[col] = df[col].apply(str)

    df.to_csv(path, index=False, quoting=csv.QUOTE_NONNUMERIC)

export_to_csv(innovi_df, OUT_DIR / "innovi_dataset_processed.csv")
export_to_csv(tactic_df, OUT_DIR / "tactic_dataset_processed.csv")

# Particiones

### Pequeña partición para test

In [28]:
small_train = innovi_df[innovi_df.geometry.notna()]
small_train = small_train.sample(5)
export_to_csv(small_train, OUT_DIR / "aoi_eda.csv")


In [24]:
small_train.yield_kg_ha.describe()

count       5.000000
mean     3531.672000
std       982.765631
min      2411.790000
25%      3126.460000
50%      3164.470000
75%      3968.040000
max      4987.600000
Name: yield_kg_ha, dtype: float64

In [14]:
innovi_df[innovi_df['geometry'].isna()]

,place_id,year,harvest_date,province,municipality,polygon,parcel,enclosure,total_enclosures,declared_area_ha,area_ha,production_kg,yield_kg_ha,alcohol_degree,geometry
1152,L510642,2021,2021-09-06,[8],[249],[1],[2],[12],1,0.10,NaN,840,NaN,11.00,None
1153,L510642,2023,2023-09-05,[8],[249],[1],[2],[12],1,0.10,NaN,1100,NaN,10.80,None
1154,L510642,2024,2024-09-07,[8],[249],[1],[2],[12],1,0.10,NaN,740,NaN,10.00,None
1180,"L510649,L510658",2021,2021-09-03,[8],[249],[1],[2],"[52, 41]",2,0.05,NaN,1540,NaN,8.70,None
1181,"L510649,L510658",2024,2024-08-28,[8],[249],[1],[2],"[52, 41]",2,0.05,NaN,700,NaN,10.00,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10574,"L64444,L64459",2021,2021-08-31,[43],[177],[9],"[134, 158]","[19, 1]",2,1.36,NaN,14780,NaN,12.22,None
10575,"L64444,L64459",2022,2022-09-14,[43],[177],[9],"[134, 158]","[19, 1]",2,1.36,NaN,20476,NaN,13.50,None
10576,"L64444,L64459",2023,2023-08-29,[43],[177],[9],"[134, 158]","[19, 1]",2,1.36,NaN,11720,NaN,13.30,None
10577,"L64444,L64459",2024,2024-08-21,[43],[177],[9],"[134, 158]","[19, 1]",2,1.36,NaN,4760,NaN,12.20,None


In [11]:
final_innovi_df

,place_id,year,harvest_date,province,municipality,polygon,parcel,enclosure,total_enclosures,declared_area_ha,area_ha,production_kg,yield_kg_ha,alcohol_degree,geometry
0,"L1100,L194",2021,2021-09-22,[8],[13],[4],[49],"[6, 10]",2,1.19,2.12,6734,3180.32,11.40,"MULTIPOLYGON (((197985.277 5068113.845, 197898..."
1,"L1100,L194",2022,2022-09-16,[8],[13],[4],[49],"[6, 10]",2,1.19,2.12,5350,2526.68,12.00,"MULTIPOLYGON (((197985.277 5068113.845, 197898..."
2,"L1100,L194",2023,2023-09-07,[8],[13],[4],[49],"[6, 10]",2,1.19,2.12,1520,717.86,12.00,"MULTIPOLYGON (((197985.277 5068113.845, 197898..."
3,"L1100,L194",2024,2024-09-30,[8],[13],[4],[49],"[6, 10]",2,1.19,2.12,2960,1397.94,12.50,"MULTIPOLYGON (((197985.277 5068113.845, 197898..."
4,L1101,2021,2021-09-06,[8],[13],[4],[50],[1],1,0.80,1.43,8810,6164.83,10.67,"POLYGON ((197818.336 5068109.654, 197795.093 5..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12074,"L763,L764",2024,2024-08-29,[43],[73],[3],[40],"[32, 51]",2,0.30,0.54,771,1431.24,14.00,"MULTIPOLYGON (((84469.288 5041681.799, 84469.8..."
12075,L767,2021,2021-09-08,[43],[56],[6],[167],[31],1,0.12,0.21,900,4266.55,12.50,"POLYGON ((92186.148 5037841.341, 92185.925 503..."
12076,L767,2022,2022-08-31,[43],[56],[6],[167],[31],1,0.12,0.21,837,3967.89,14.00,"POLYGON ((92186.148 5037841.341, 92185.925 503..."
12077,L767,2023,2023-08-30,[43],[56],[6],[167],[31],1,0.12,0.21,653,3095.62,17.00,"POLYGON ((92186.148 5037841.341, 92185.925 503..."


In [12]:
aoi_df[
    (aoi_df.provincia == 43) &
    (aoi_df.municipio == 22) &
    (aoi_df.poligono == 2) &
    (aoi_df.parcela == 18)
]

,id_lugar,provincia,municipio,agregado,zona,poligono,parcela,recinto,geometry
